# B05C — Evaluasi test dari checkpoint B05B Fixed25
Jalankan Cell 1–7 dari atas ke bawah pada runtime Colab GPU baru. Ubah **DATASET** saja untuk mengevaluasi dataset berikutnya. Enam run per dataset: dua supervised controls dan empat pasangan Hinton KD. Tidak ada pemanggilan training atau pembuatan cache Teacher.

Protokol training dibaca dari Drive dan tidak ditulis ulang. Semua checkpoint ditetapkan berdasarkan validation sebelum inferensi test. Threshold tetap 0,5; seed 42; alpha 0,5 dan T=2 pada KD. Hasil disimpan terpisah di `b05c_fixed25_test_v1`. Saat dilanjutkan, hasil test yang sudah lengkap diverifikasi hash-nya dan digunakan kembali; run yang belum selesai dievaluasi.

**Interpretasi:** test B03 pernah diperiksa, sehingga bukan holdout yang belum pernah dilihat. Hasil satu seed bersifat deskriptif. INbreast: BI-RADS 1/2 vs 5/6, bukan label patologi; MIAS: abnormal-only; CMMD: sertakan macro-F1 dan specificity. Tidak memilih threshold atau pasangan baru berdasarkan hasil test.

## Cell 1 — Runtime dan Drive

In [ ]:
import os,sys,subprocess,importlib.metadata as metadata
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'

import tensorflow as tf
tf_minor='.'.join(tf.__version__.split('.')[:2])
try: installed=metadata.version('tf-keras')
except metadata.PackageNotFoundError: installed=''
if '.'.join(installed.split('.')[:2])!=tf_minor:
    if 'tf_keras' in sys.modules:raise RuntimeError('Restart runtime before changing tf-keras')
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-deps',f'tf-keras~={tf_minor}.0'])

import tf_keras as keras
from pathlib import Path
from google.colab import drive
from IPython.display import display
gpus=tf.config.list_physical_devices('GPU')
if not gpus:raise RuntimeError('Select GPU runtime in Colab')
for gpu in gpus:tf.config.experimental.set_memory_growth(gpu,True)
keras.mixed_precision.set_global_policy('float32')
tf.config.experimental.enable_op_determinism()

print('TensorFlow:',tf.__version__,'tf-keras:',keras.__version__,'GPU:',gpus)
drive.mount('/content/drive')

## Cell 2 — Pilih dataset

In [ ]:
RESEARCH=Path('/content/drive/MyDrive/Research')
DATASET='cbis_ddsm'  # cbis_ddsm | cmmd | inbreast | mias
EVALUATE_TEST=True  # False: hanya audit sumber dan restore validation
DOWNLOAD_REPORT=True
SOURCE_ROOT=RESEARCH/'outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1'
TEST_ROOT=RESEARCH/'outputs/benchmark_4datasets/b05c_fixed25_test_v1'
EXPERIMENT_IDS={'cbis_ddsm':'7a4bf6d1e3f6','cmmd':'f81ede1d807a',
                'inbreast':'3f2fbdc2b913','mias':'672f572d6c33'}
B04A_REGISTRY=RESEARCH/'outputs/benchmark_4datasets/b04_preprocessing_v1/1dcb7ddadfe6/registry.json'
LOCAL_IMAGES=Path('/content/b05c_images')
if DATASET not in EXPERIMENT_IDS:raise ValueError('Unknown dataset')
if SOURCE_ROOT.resolve()==TEST_ROOT.resolve() or SOURCE_ROOT.resolve() in TEST_ROOT.resolve().parents:
    raise ValueError('Output test harus terpisah dari training')
SOURCE_DIR=SOURCE_ROOT/DATASET/EXPERIMENT_IDS[DATASET]

## Cell 3 — Fungsi inferensi dan audit
Tidak mengandung fungsi training. Arsitektur, preprocessing dan ekspor metrik menggunakan implementasi notebook sumber.

In [ ]:
import os, io, gc, json, time, hashlib, zipfile, shutil, platform

from pathlib import Path

import numpy as np

import pandas as pd

import tensorflow as tf

import tf_keras as keras

from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,
    roc_auc_score, average_precision_score, confusion_matrix, balanced_accuracy_score)

REGISTRY = {
    'vgg16': ('teacher','VGG16','vgg16'),
    'resnet50': ('teacher','ResNet50','resnet'),
    'inception_v3': ('teacher','InceptionV3','inception_v3'),
    'regnet_y_8gf': ('teacher','RegNetY080',None),
    'densenet121': ('student','DenseNet121','densenet'),
    'mobilenet_v2': ('student','MobileNetV2','mobilenet_v2'),
    'mobilenet_v3': ('student', 'MobileNetV3Large', None),
    'nasnet_mobile': ('student','NASNetMobile','nasnet'),
}

def json_safe(value):
    if isinstance(value, dict): return {str(k):json_safe(v) for k,v in value.items()}
    if isinstance(value, (tuple,list)): return [json_safe(v) for v in value]
    if isinstance(value, np.generic): return json_safe(value.item())
    if isinstance(value, float) and not np.isfinite(value): return None
    return value

def save_json(path, obj):
    path=Path(path); tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(json_safe(obj),indent=2),encoding='utf-8'); os.replace(tmp,path)

def file_sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''): h.update(b)
    return h.hexdigest()

def metrics(y, p):
    y=np.asarray(y,dtype=int); p=np.asarray(p,dtype=float).reshape(-1)
    if len(y)!=len(p) or not np.isfinite(p).all() or ((p<0)|(p>1)).any():
        raise ValueError('Invalid predictions')
    pred=(p>=.5).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return dict(precision=precision_score(y,pred,zero_division=0),
        recall=recall_score(y,pred,zero_division=0), f1=f1_score(y,pred,zero_division=0),
        macro_f1=f1_score(y,pred,labels=[0,1],average='macro',zero_division=0),
        accuracy=accuracy_score(y,pred), balanced_accuracy=balanced_accuracy_score(y,pred),
        specificity=tn/(tn+fp) if tn+fp else np.nan,
        roc_auc=roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan,
        average_precision=average_precision_score(y,p) if len(np.unique(y))==2 else np.nan,
        tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp),threshold=.5,images=len(y))

def make_dataset(frame, image_dir, key, batch, seed, epoch=0, training=False, size=224):
    paths=[str(image_dir/p) for p in frame.cache_member]
    labels=frame.label.to_numpy(dtype=np.float32)
    d=tf.data.Dataset.from_tensor_slices((paths,labels))
    if training:
        d=d.shuffle(len(frame),seed=seed+epoch,reshuffle_each_iteration=False)
    preprocess_module=REGISTRY[key][2]
    def load(index, pair):
        path,label=pair
        x=tf.image.decode_png(tf.io.read_file(path),channels=3)
        x=tf.ensure_shape(tf.cast(x,tf.float32),[size,size,3])
        if training:
            # Stateless horizontal flip: image identity is not a laterality target.
            x=tf.image.stateless_random_flip_left_right(x,seed=tf.stack([
                tf.cast(seed+epoch*1009,tf.int32),tf.cast(index,tf.int32)]))
        if preprocess_module:
            x=getattr(keras.applications,preprocess_module).preprocess_input(x)
        # MobileNetV3 / RegNet contain their own rescaling and consume [0,255].
        return x,tf.reshape(label,[1])
    options=tf.data.Options(); options.experimental_deterministic=True
    options.threading.private_threadpool_size=2
    return d.enumerate().map(load,num_parallel_calls=2,deterministic=True).batch(batch).with_options(options).prefetch(1)

def load_ready_dataset(ready_path, local_root):
    ready_path=Path(ready_path); meta=json.loads(ready_path.read_text()); folder=ready_path.parent
    for filename,digest in meta['file_sha256'].items():
        if file_sha(folder/filename)!=digest: raise RuntimeError('B02 cache changed: '+filename)
    frames={s:pd.read_csv(folder/(s+'.csv'),dtype={'patient_id':str,'sample_id':str})
            for s in ['train','validation','test']}
    all_d=pd.concat(frames.values(),ignore_index=True)
    assert not all_d.sample_id.duplicated().any()
    assert all_d.groupby('patient_id').split.nunique().max()==1
    assert all(set(d.label)=={0,1} for d in frames.values())
    image_dir=Path(local_root)/meta['dataset']/meta['file_sha256']['images.zip'][:16]
    image_dir.mkdir(parents=True,exist_ok=True)
    marker=image_dir/'EXTRACTED.json'
    expected=meta['file_sha256']['images.zip']
    valid=marker.exists() and json.loads(marker.read_text()).get('sha256')==expected
    if not valid or not all((image_dir/f).is_file() for f in all_d.cache_member):
        local_zip=image_dir/'images.zip'
        shutil.copyfile(folder/'images.zip',local_zip)
        if file_sha(local_zip)!=expected: raise RuntimeError('Local cache copy corrupted')
        with zipfile.ZipFile(local_zip) as z:
            for member in all_d.cache_member:
                if Path(member).name!=member: raise ValueError('Unsafe cache member')
                (image_dir/member).write_bytes(z.read(member))
        save_json(marker,{'sha256':expected})
    # Audit the actual extracted model inputs, including reused local cache.
    from PIL import Image
    for row in all_d.itertuples():
        with Image.open(image_dir/row.cache_member) as im:
            a=np.asarray(im)
            if a.shape!=(meta['image_size'],meta['image_size']): raise ValueError('Unexpected image size')
            if hashlib.sha256(a.tobytes()).hexdigest()!=row.processed_pixel_sha256:
                raise RuntimeError('Corrupted local image cache')
    return frames,image_dir,meta

def build_model(key, size=224, seed=42, weights='imagenet', learning_rate=1e-5, stage='finetune'):
    keras.utils.set_random_seed(seed)
    _, constructor, _ = REGISTRY[key]
    base = getattr(keras.applications, constructor)(include_top=False, weights=weights,
        input_shape=(size,size,3), pooling='avg')
    base.trainable = stage == 'finetune'
    for layer in base.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False
            layer.fused = False
    inp = keras.Input((size,size,3), name='preprocessed_image')
    x = base(inp, training=False)
    x = keras.layers.Dropout(.2, seed=seed, name='head_dropout')(x)
    output = keras.layers.Dense(1, activation='sigmoid', dtype='float32', name='probability')(x)
    model = keras.Model(inp, output, name=key)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.BinaryCrossentropy())
    model.optimizer.build(model.trainable_variables)
    return model

def weight_snapshot(model):
    # Explicitly exclude optimizer: head and fine-tuning have different variable sets.
    base = next(layer for layer in model.layers if isinstance(layer, keras.Model))
    return tf.train.Checkpoint(backbone=base, classifier=model.get_layer('probability'))

def probability_diagnostics(y, p):
    p=np.asarray(p,dtype=np.float64).reshape(-1)
    q=np.clip(p,1e-7,1-1e-7); y=np.asarray(y,dtype=np.float64)
    return dict(probability_min=float(p.min()), probability_max=float(p.max()),
        probability_std=float(p.std()), positive_prediction_fraction=float((p>=.5).mean()),
        validation_bce_unweighted=float(-(y*np.log(q)+(1-y)*np.log(1-q)).mean()))

def preprocessing_audit(frames, image_dir, meta, key, protocol):
    from PIL import Image
    subset=frames['train'].iloc[:protocol['batch_size']]
    raw=np.stack([np.asarray(Image.open(image_dir/name).convert('RGB'),dtype=np.float32)
                  for name in subset.cache_member])
    ds=make_dataset(subset,image_dir,key,protocol['batch_size'],protocol['training_seed'],size=meta['image_size'])
    x,y=next(iter(ds)); actual=x.numpy()
    if key in ['vgg16','resnet50']:
        expected=raw[...,::-1]-np.array([103.939,116.779,123.68],dtype=np.float32)
    elif key=='densenet121':
        expected=(raw/255.-np.array([.485,.456,.406],dtype=np.float32))/np.array([.229,.224,.225],dtype=np.float32)
    elif key in ['inception_v3','mobilenet_v2','nasnet_mobile']:
        expected=raw/127.5-1.
    else: expected=raw  # RegNet and MobileNetV3 use internal preprocessing.
    np.testing.assert_allclose(actual,expected,rtol=1e-6,atol=2e-5)
    np.testing.assert_equal(y.numpy().reshape(-1),subset.label.to_numpy())
    return dict(model=key,constructor=REGISTRY[key][1],preprocessing=REGISTRY[key][2],
        checked_split='train',checked_images=len(subset),input_min=float(actual.min()),
        input_max=float(actual.max()),input_std=float(actual.std()),formula_check='passed')

def environment_record():
    return dict(python=platform.python_version(),tensorflow=tf.__version__,tf_keras=keras.__version__,
        gpu=[tf.config.experimental.get_device_details(g) for g in tf.config.list_physical_devices('GPU')])

import warnings
from sklearn.metrics import ConfusionMatrixDisplay
def object_sha(value):
    return hashlib.sha256(json.dumps(json_safe(value), sort_keys=True).encode()).hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text())

def guarded_json(path, value):
    if Path(path).exists() and read_json(path) != json_safe(value):
        raise RuntimeError('Existing artifact has a different configuration: '+str(path))
    save_json(path, value)

def check_frame(frame, reference):
    columns=['sample_id','patient_id','label']
    if frame.sample_id.duplicated().any() or set(frame.sample_id)!=set(reference.sample_id):
        raise RuntimeError('Prediction IDs differ from the frozen cohort')
    aligned=frame.set_index('sample_id').loc[reference.sample_id].reset_index()
    for col in columns:
        if not np.array_equal(aligned[col].to_numpy(), reference[col].to_numpy()):
            raise RuntimeError('Prediction identity/label mismatch: '+col)
    return aligned

def export_result(model,key,frame,image_dir,meta,folder,split,protocol,extra):
    ds=make_dataset(frame,image_dir,key,protocol['batch_size'],protocol['training_seed'],size=meta['image_size'])
    p=model.predict(ds,verbose=0).reshape(-1)
    result=dict(metrics(frame.label,p),model=key,dataset=meta['dataset'],training_seed=protocol['training_seed'],
        split=split,role='student',**extra)
    table=frame[['sample_id','patient_id','label']].copy()
    table['probability']=p.astype(np.float64);table['prediction']=(p>=.5).astype(int)
    for k in ['model','dataset','training_seed','split','teacher','method']:table[k]=result[k]
    table.to_csv(folder/f'{split}_predictions.csv',index=False);save_json(folder/f'{split}_metrics.json',result)
    import matplotlib.pyplot as plt
    names=['BI-RADS 1/2','BI-RADS 5/6'] if meta['dataset']=='inbreast' else ['Benign','Malignant']
    fig,ax=plt.subplots(figsize=(5.6,5))
    ConfusionMatrixDisplay(np.array([[result['tn'],result['fp']],[result['fn'],result['tp']]]),display_labels=names).plot(
        ax=ax,cmap='Blues',values_format='d',im_kw={'vmin':0,'vmax':int(frame.label.value_counts().max())})
    ax.set_title(f"{key} | {result['method']} | {result['teacher']}\n{meta['dataset']} {split}; seed {protocol['training_seed']}; threshold 0.5",fontsize=9)
    fig.tight_layout();fig.savefig(folder/f'{split}_confusion_matrix.png',dpi=180);plt.close(fig)
    return result

def compare_controls(records,root,split='validation'):
    table=pd.DataFrame(records);columns=['precision','recall','f1','macro_f1','accuracy','balanced_accuracy','specificity','roc_auc','average_precision']
    rows=[]
    for record in records:
        if record['teacher']=='none':continue
        control=table[(table.model==record['model'])&(table.teacher=='none')]
        if len(control)!=1:raise RuntimeError('One matched supervised control is required')
        base=control.iloc[0];row={k:record[k] for k in ['dataset','training_seed','model','teacher','method','split']}
        for metric in columns:row['delta_'+metric]=record[metric]-base[metric]
        rows.append(row)
    result=pd.DataFrame(rows);result.to_csv(Path(root)/f'{split}_kd_vs_control.csv',index=False)
    return result

## Cell 4 — Audit protokol, data dan seluruh checkpoint sebelum test

In [ ]:
PROTOCOL=read_json(SOURCE_DIR/'protocol.json')
if object_sha(PROTOCOL)[:12]!=EXPERIMENT_IDS[DATASET]:raise RuntimeError('Protocol ID mismatch')
expected=dict(dataset=DATASET,training_seed=42,teacher_seed=42,early_stopping=False,
              head_epochs=5,finetune_epochs=25,batch_size=8,image_size=512,
              variant='full_512',alpha=.5,temperature=2.,threshold=.5)
for key,value in expected.items():
    if PROTOCOL.get(key)!=value:raise RuntimeError('Frozen protocol mismatch: '+key)
for name,key in [('b04_compat.py','b04_compat_sha256'),('b05_hinton_engine.py','engine_sha256')]:
    if file_sha(SOURCE_DIR/name)!=PROTOCOL[key]:raise RuntimeError('Source hash mismatch: '+name)
if PROTOCOL['b04_compat_sha256']!='2822801a4f73ab90d6df8215d12fad03ea832cbc2300bdeb36d729d66ab66297':
    raise RuntimeError('Unexpected model implementation')
if PROTOCOL['engine_sha256']!='259eb201e72cc0479ad1db184716d9bad9a677ade44b7d5892de3b95f2c67aa9':
    raise RuntimeError('Unexpected training engine')
entry=read_json(B04A_REGISTRY)[DATASET]['full_512']
if entry['status']!='READY':raise RuntimeError('Cache not READY')
FRAMES,IMAGE_DIR,DATA_META=load_ready_dataset(Path(entry['path'])/'READY.json',LOCAL_IMAGES)
for key,pkey in [('file_sha256','data_sha256'),('cohort_sha256','cohort_sha256'),
                 ('preprocessing_source_sha256','preprocessing_source_sha256'),('split_seed','split_seed')]:
    if DATA_META[key]!=PROTOCOL[pkey]:raise RuntimeError('Data provenance mismatch: '+key)
if DATA_META['dataset']!=DATASET or DATA_META['image_size']!=512:raise RuntimeError('Wrong dataset')
RUNS=[]; SOURCE_HASHES={}; initials={}
def track(path):
    path=Path(path); SOURCE_HASHES[str(path)]=file_sha(path); return SOURCE_HASHES[str(path)]
track(SOURCE_DIR/'protocol.json')
for student in ['mobilenet_v2','nasnet_mobile']:
    for teacher in ['none','vgg16','densenet121']:
        folder=SOURCE_DIR/(student+'__from__'+teacher)/'seed_42'
        cfg=read_json(folder/'config.json'); done=read_json(folder/'COMPLETE.json')
        for key,val in PROTOCOL.items():
            if cfg.get(key)!=val:raise RuntimeError('Run protocol differs: '+str(folder)+' '+key)
        if cfg['student']!=student or cfg['teacher']!=teacher:raise RuntimeError('Wrong pair')
        if track(folder/'config.json')!=done['config_sha256']:raise RuntimeError('Config changed')
        track(folder/'COMPLETE.json'); track(folder/'validation_predictions.csv')
        val=read_json(folder/'validation_metrics.json'); track(folder/'validation_metrics.json')
        hist=pd.read_csv(folder/'history.csv'); track(folder/'history.csv')
        if val['epochs_completed']!=30 or len(hist)!=30:raise RuntimeError('Incomplete training')
        for stage,budget in [('head',5),('finetune',25)]:
            if len(pd.read_csv(folder/stage/'history.csv'))!=budget:raise RuntimeError('Incomplete stage')
        initial=read_json(folder/'head/initialization.json')
        if teacher=='none':initials[student]=initial
        elif initial!=initials[student]:raise RuntimeError('Unmatched initialization')
        prefix=Path(done['best_checkpoint'])
        if folder.resolve() not in prefix.resolve().parents:raise RuntimeError('Checkpoint outside run')
        parts=sorted(prefix.parent.glob(prefix.name+'.*'))
        if not Path(str(prefix)+'.index').is_file() or not any('.data-' in x.name for x in parts):
            raise FileNotFoundError('Checkpoint weights missing: '+str(prefix))
        for part in parts:track(part)
        RUNS.append(dict(student=student,teacher=teacher,folder=folder,checkpoint=str(prefix),config=cfg))
LOCK=dict(version='b05c_fixed25_test_v1',dataset=DATASET,source_experiment=EXPERIMENT_IDS[DATASET],
          source_sha256=SOURCE_HASHES,data_sha256=DATA_META['file_sha256'],threshold=.5,
          inference_batch_size=8,validation_restore_atol=1e-5,validation_restore_rtol=1e-5,
          tensorflow=tf.__version__,tf_keras=keras.__version__,precision='float32',
          test_status='B03 previously inspected; no test-based tuning in B05')
EVAL_DIR=TEST_ROOT/DATASET/EXPERIMENT_IDS[DATASET]
EVAL_DIR.mkdir(parents=True,exist_ok=True)
guarded_json(EVAL_DIR/'evaluation_lock.json',LOCK)
save_json(EVAL_DIR/'training_protocol_reference.json',PROTOCOL)
save_json(EVAL_DIR/'environment.json',environment_record())
AUDIT=pd.DataFrame([dict(split=s,images=len(f),patients=f.patient_id.nunique(),
    negative=int((f.label==0).sum()),positive=int((f.label==1).sum())) for s,f in FRAMES.items()])
AUDIT.to_csv(EVAL_DIR/'split_audit.csv',index=False); display(AUDIT)
print('Enam checkpoint terkunci. Output:',EVAL_DIR)

## Cell 5 — Restore validation seluruh model
Seluruh model harus lolos pemeriksaan sebelum satu pun inferensi test. Perbedaan probabilitas hanya ditoleransi sampai `atol=1e-5, rtol=1e-5`, dan prediksi kelas harus identik. Jika gagal, kirim audit; jangan memperbesar toleransi tanpa diagnosis.

In [ ]:
def restore(run):
    keras.backend.clear_session();gc.collect()
    model=build_model(run['student'],512,42,weights=None,stage='head')
    weight_snapshot(model).restore(run['checkpoint']).assert_existing_objects_matched()
    return model
RESTORE_AUDITS=[]
for run in RUNS:
    model=restore(run)
    saved=pd.read_csv(run['folder']/'validation_predictions.csv',
                      dtype={'sample_id':str,'patient_id':str},float_precision='round_trip')
    saved=check_frame(saved,FRAMES['validation'])
    p=model.predict(make_dataset(FRAMES['validation'],IMAGE_DIR,run['student'],8,42,size=512),verbose=0).reshape(-1)
    oldp=saved.probability.to_numpy(float)
    close=np.allclose(p,oldp,atol=1e-5,rtol=1e-5)
    same=np.array_equal(p>=.5,oldp>=.5)
    RESTORE_AUDITS.append(dict(model=run['student'],teacher=run['teacher'],passed=bool(close and same),
        max_abs_difference=float(np.max(np.abs(p-oldp))),class_disagreements=int(np.sum((p>=.5)!=(oldp>=.5)))))
    del model;keras.backend.clear_session();gc.collect()
restore_table=pd.DataFrame(RESTORE_AUDITS)
restore_table.to_csv(EVAL_DIR/'validation_restore_audit.csv',index=False);display(restore_table)
if not restore_table.passed.all():raise RuntimeError('Restore audit failed. Test belum dijalankan; kirim validation_restore_audit.csv.')
RESTORE_PASSED=True

## Cell 6 — Evaluasi test enam run dan selisih KD terhadap control
Semua pasangan dilaporkan. Tidak ada tuning threshold atau training ulang. Jika terputus, jalankan dari atas dengan dataset yang sama.

In [ ]:
if not RESTORE_PASSED:raise RuntimeError('Run Cell 5 first')
RESULTS=[]
if EVALUATE_TEST:
    for path,digest in SOURCE_HASHES.items():
        if file_sha(path)!=digest:raise RuntimeError('Source changed after freeze: '+path)
    for run in RUNS:
        out=EVAL_DIR/(run['student']+'__from__'+run['teacher'])
        out.mkdir(exist_ok=True)
        completion=out/'EVALUATION_COMPLETE.json'
        if completion.exists():
            record=read_json(completion)
            if record['lock_sha256']!=object_sha(LOCK):raise RuntimeError('Result lock mismatch')
            for name,digest in record['output_sha256'].items():
                if file_sha(out/name)!=digest:raise RuntimeError('Saved test output changed: '+name)
            saved=pd.read_csv(out/'test_predictions.csv',dtype={'sample_id':str,'patient_id':str},float_precision='round_trip')
            check_frame(saved,FRAMES['test'])
            result=read_json(out/'test_metrics.json')
        else:
            print('TEST',DATASET,run['student'],'<-',run['teacher'],flush=True)
            model=restore(run)
            result=export_result(model,run['student'],FRAMES['test'],IMAGE_DIR,DATA_META,out,'test',PROTOCOL,
                dict(teacher=run['teacher'],method=run['config']['method'],run_folder=str(run['folder'])))
            del model;keras.backend.clear_session();gc.collect()
            names=['test_predictions.csv','test_metrics.json','test_confusion_matrix.png']
            save_json(completion,dict(lock_sha256=object_sha(LOCK),output_sha256={n:file_sha(out/n) for n in names}))
        RESULTS.append(result)
    summary=pd.DataFrame(RESULTS)
    summary.to_csv(EVAL_DIR/'test_summary.csv',index=False)
    display(summary[['dataset','model','teacher','precision','recall','f1','macro_f1','specificity','accuracy','roc_auc']].round(4))
    display(compare_controls(RESULTS,EVAL_DIR,split='test').round(4))
    for path,digest in SOURCE_HASHES.items():
        if file_sha(path)!=digest:raise RuntimeError('Source changed during evaluation: '+path)
    print('Selesai: enam run; checkpoint dan protokol training tidak berubah.')
else:
    print('Audit selesai. EVALUATE_TEST=False: test belum dievaluasi.')

## Cell 7 — Unduh laporan
ZIP berisi hasil test, prediksi per citra, confusion matrix Blues, selisih KD–control, audit restore dan lock sumber. Bobot tetap di folder training. Ganti DATASET pada Cell 2 dan jalankan Cell 2–7 untuk dataset berikutnya; Cell 1 cukup sekali dalam runtime yang sama.

In [ ]:
REPORT_PATH=EVAL_DIR/f'B05C_Fixed25_{DATASET}_seed_42_{EXPERIMENT_IDS[DATASET]}_report.zip'
with zipfile.ZipFile(REPORT_PATH,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(EVAL_DIR.rglob('*')):
        if path.is_file() and path.suffix in ['.json','.csv','.png']:
            archive.write(path,str(path.relative_to(EVAL_DIR)))
print('Report:',REPORT_PATH)
if DOWNLOAD_REPORT:
    from google.colab import files
    files.download(str(REPORT_PATH))